In [13]:
import pandas as pd
import numpy as np
from thefuzz import fuzz
import re

# 1. Load the WRDS CSV (Pre-filtered for Relevance >= 100 and ESD >= 90)
df = pd.read_csv('raw ravenpack.csv')

/var/folders/w3/6myr5tjs6ls3zn1g3ywbc29r0000gn/T/ipykernel_17774/3926018667.py:7: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('raw ravenpack.csv')


In [14]:
df

,rp_entity_id,rpa_date_utc,timestamp_utc,rp_story_id,entity_name,event_sentiment_score,event_relevance,event_similarity_days,category,news_type,headline,ticker,cusip
0,01CB4E,2023-04-01,2023-04-01 06:53:51.564,E6DB53D9D0026D6BE19C24CBDD660B85,Prairie Provident Resources Inc.,0.00,100,365.00000,earnings,PRESS-RELEASE,Prairie Provident Resources Announces Fourth Q...,PPR,73965Q888
1,01CB4E,2023-04-01,2023-04-01 06:53:51.564,E6DB53D9D0026D6BE19C24CBDD660B85,Prairie Provident Resources Inc.,0.00,100,365.00000,earnings,PRESS-RELEASE,Prairie Provident Resources Announces Fourth Q...,PPR.T,73965Q888
2,01CB4E,2023-04-01,2023-04-01 06:53:51.564,E6DB53D9D0026D6BE19C24CBDD660B85,Prairie Provident Resources Inc.,0.00,100,365.00000,earnings,PRESS-RELEASE,Prairie Provident Resources Announces Fourth Q...,PRPRF,73965Q888
3,08B4B5,2023-04-01,2023-04-01 01:33:04.437,D9520DD4170FFDF8137787E66AAF3A7A,Roivant Sciences Ltd.,0.00,100,365.00000,insider-surrender,TABULAR-MATERIAL,"CFO Pulik Surrenders 1,639 Of Roivant Sciences...",ROIV,G76279101
4,0E959A,2023-04-01,2023-04-01 09:22:18.453,F5E13FFD7783E1EBE8A5460B8A0C60CE,Sat Industries Ltd.,0.59,100,365.00000,ipo-considered,PRESS-RELEASE,BSE: SAT Industries Ltd. - Proposed Initial Pu...,511076,Y7532P109
...,...,...,...,...,...,...,...,...,...,...,...,...,...
45105142,992E92,2025-01-02,2025-01-02 22:51:37.761,314F6DD017447DC3ACF567BB7FCFBB03,SEI Investments Co.,0.00,100,365.00000,insider-gift,TABULAR-MATERIAL,"Dir Romeo Gifts 6,300 Of SEI Investments Co >SEIC",SEIC,784117103
45105143,992E92,2025-01-02,2025-01-02 22:51:37.761,314F6DD017447DC3ACF567BB7FCFBB03,SEI Investments Co.,0.00,100,365.00000,insider-gift,TABULAR-MATERIAL,"Dir Romeo Gifts 6,300 Of SEI Investments Co >SEIC",SEIC,784117103
45105144,996629,2025-01-02,2025-01-02 21:42:29.641,F2680773270ECC2CC2DBD0EE9DDA7B43,Rush Street Interactive Inc.,-0.48,100,128.08545,insider-sell-registration,RNS-SEC144,"Officer Roosileht Registers 70,000 Of Rush Str...",RSI,782011100
45105145,9A3FF4,2025-01-02,2025-01-02 22:24:19.302,69CDF79EF94905AE8673EE5C56C54795,Allegion PLC,-0.11,100,365.00000,stock-loss,FULL-ARTICLE,"Allegion PLC Stock Falls Thursday, Underperfor...",ALLE,G0176J109


In [15]:
# Standardize columns to lowercase
df.columns = df.columns.str.lower()

# 2. Basic Filters
# Keep only full articles and press releases
if 'news_type' in df.columns:
    df = df[df['news_type'].isin(['FULL-ARTICLE', 'PRESS-RELEASE'])].copy()

# Drop stock-gain and stock-loss categories
if 'category' in df.columns:
    df = df[~df['category'].str.lower().isin(['stock-gain', 'stock-loss'])].copy()

# Drop 'SCHEDULES'
df = df[~df['headline'].str.contains('SCHEDULES', case=False, na=False)].copy()

# 3. Headline Text Cleaning
# Remove "Press Release: ", remove digit patterns like " -1- ", and strip whitespace
df['clean_headline'] = df['headline'].str.replace('Press Release: ', '', regex=False)
df['clean_headline'] = df['clean_headline'].str.replace(r' -\d+-', '', regex=True)
df['clean_headline'] = df['clean_headline'].str.strip()

# Drop rows where headline became completely empty
df = df[df['clean_headline'] != ""].copy()
df = df.dropna(subset=['clean_headline']).copy()

# 4. Timezone and the 4:00 PM Trading Shift (For Close-to-Close Strategies)
df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], format='mixed', utc=True)
df['et_time'] = df['timestamp_utc'].dt.tz_convert('America/New_York')

# Extract the raw calendar date and the hour
df['raw_date'] = df['et_time'].dt.date
df['hour_et'] = df['et_time'].dt.hour

# Shift news arriving at or after 4:00 PM (16:00) to the next day
df['date'] = np.where(
    df['hour_et'] >= 16,
    df['raw_date'] + pd.Timedelta(days=1),
    df['raw_date']
)

# Fix weekends (If shifted date lands on Saturday(5) or Sunday(6), move to Monday)
df['date'] = pd.to_datetime(df['date'])
df['date'] = df['date'].apply(
    lambda x: x + pd.Timedelta(days=2) if x.weekday() == 5 
    else x + pd.Timedelta(days=1) if x.weekday() == 6 
    else x
)

# Drop the helper columns
df = df.drop(columns=['raw_date', 'hour_et'])

# 5. Exact Deduplication
# Drop duplicate story IDs (if available)
if 'rp_story_id' in df.columns:
    df = df.drop_duplicates(subset=['rp_story_id', 'rp_entity_id'], keep='first')

# Exact match dedup on ticker, date, and clean headline
df = df.drop_duplicates(subset=['ticker', 'date', 'clean_headline'], keep='first')

# 6. Fuzzy Deduplication (60% similarity threshold)
print("Running fuzzy deduplication (this may take a moment)...")

def is_unique_fuzzy(group_df, threshold=60):
    """Marks rows as unique if they are less than 60% similar to previously approved rows in the group"""
    if len(group_df) <= 1:
        return [True] * len(group_df)
    
    headlines = group_df['clean_headline'].tolist()
    keep_flags = [True]
    
    for i in range(1, len(headlines)):
        is_distinct = True
        # Compare current headline against all previously kept headlines
        for j in range(i):
            if keep_flags[j]: 
                # fuzz.ratio returns 0-100
                similarity = fuzz.ratio(headlines[i], headlines[j])
                if similarity > threshold:
                    is_distinct = False
                    break
        keep_flags.append(is_distinct)
    return keep_flags

# Apply fuzzy deduplication grouped by exact timestamp and ticker
df = df.sort_values(by=['ticker', 'timestamp_utc'])
df['is_unique'] = df.groupby(['ticker', 'timestamp_utc'], group_keys=False).apply(
    lambda g: pd.Series(is_unique_fuzzy(g), index=g.index)
)

# Filter to only keep unique rows and drop the helper column
df_clean = df[df['is_unique'] == True].drop(columns=['is_unique'])

# 7. Final Output
# Sort for final clean dataset
df_clean = df_clean.sort_values(by=['ticker', 'date', 'timestamp_utc'])

print(f"Final dataset rows: {len(df_clean)}")
df_clean.to_csv('ravenpack_fully_cleaned.csv', index=False)

Running fuzzy deduplication (this may take a moment)...


/var/folders/w3/6myr5tjs6ls3zn1g3ywbc29r0000gn/T/ipykernel_17774/1283749010.py:86: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['is_unique'] = df.groupby(['ticker', 'timestamp_utc'], group_keys=False).apply(


Final dataset rows: 288483


In [16]:
import wrds

# 1. Connect to the WRDS Database
# If running locally, this will prompt you for your WRDS username and password
print("Connecting to WRDS...")
db = wrds.Connection()

# 2. Pull the Historical S&P 500 Constituents
# We use the CRSP Daily S&P 500 List (v2 format) and filter for recent members
print("Pulling S&P 500 historical constituents...")
sp500_query = """
    SELECT permno, mbrstartdt AS start_date, mbrenddt AS end_date
    FROM crsp_a_indexes.dsp500list_v2
    WHERE mbrenddt >= '2023-01-01'
"""
sp500_df = db.raw_sql(sp500_query, date_cols=['start_date', 'end_date'])

# If the company is still in the index today, WRDS leaves the end date blank or assigns a future date.
# We fill those blanks with today's date so the pandas date filter works.
sp500_df['end_date'] = sp500_df['end_date'].fillna(pd.to_datetime('today'))


# 1. Hardcode the exact RavenPack table based on your server's printout
mapping_table = 'rpna.wrds_rpa_company_names'
print(f"Using RavenPack mapping table: {mapping_table}")

# 2. Check for CRSP V2 vs Legacy
crsp_tables = db.list_tables(library='crsp')
if 'stksecurityinfohist' in crsp_tables:
    crsp_ident = 'crsp.stksecurityinfohist' # CRSP V2
else:
    crsp_ident = 'crsp.dse' # CRSP Legacy

print(f"Using CRSP identifier table: {crsp_ident}")

# 3. Execute the Link Query
print("Pulling the RavenPack to CRSP PERMNO crosswalk...")
link_query = f"""
    SELECT DISTINCT a.permno, a.ticker AS crsp_ticker, b.rp_entity_id
    FROM {crsp_ident} AS a
    INNER JOIN {mapping_table} AS b
    ON a.cusip = substr(b.cusip, 1, 8)
    WHERE a.cusip IS NOT NULL
"""

link_df = db.raw_sql(link_query)
print(f"Successfully mapped {len(link_df)} entities!")

rp_df = df_clean.copy()
rp_df['date'] = pd.to_datetime(rp_df['date'])

# 5. Execute the Merges
print("Merging and filtering for survivorship bias...")

# Merge 1: Attach the PERMNO to your RavenPack headlines
rp_with_permno = rp_df.merge(link_df, on='rp_entity_id', how='inner')

# Merge 2: Attach the S&P 500 tenure dates
merged_df = rp_with_permno.merge(sp500_df, on='permno', how='inner')

# 6. Apply Survivorship Bias Filter
# Only keep headlines published between the day the company joined the index and the day it left
final_sp500_df = merged_df[
    (merged_df['date'] >= merged_df['start_date']) & 
    (merged_df['date'] <= merged_df['end_date'])
].copy()

# 7. Final Cleanup
final_sp500_df = final_sp500_df.drop(columns=['start_date', 'end_date'])

# Drop the messy global ticker that came from RavenPack
if 'ticker' in final_sp500_df.columns:
    final_sp500_df = final_sp500_df.drop(columns=['ticker'])

# Rename the clean U.S. CRSP ticker to be your new main ticker column
final_sp500_df = final_sp500_df.rename(columns={'crsp_ticker': 'ticker'})

# Deduplicate one last time based on the PERMNO
final_sp500_df = final_sp500_df.drop_duplicates(subset=['permno', 'date', 'clean_headline'], keep='first')

# Sort chronologically
final_sp500_df = final_sp500_df.sort_values(by=['permno', 'date', 'timestamp_utc'])

print(f"Final S&P 500 constituent rows: {len(final_sp500_df)}")
final_sp500_df.to_csv('ravenpack_sp500_permno_final.csv', index=False)

# Safely close the WRDS connection
db.close()

Connecting to WRDS...
WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Pulling S&P 500 historical constituents...
Using RavenPack mapping table: rpna.wrds_rpa_company_names
Using CRSP identifier table: crsp.stksecurityinfohist
Pulling the RavenPack to CRSP PERMNO crosswalk...
Successfully mapped 22376 entities!
Merging and filtering for survivorship bias...
Final S&P 500 constituent rows: 44841


In [17]:
final_sp500_df = pd.read_csv('ravenpack_sp500_permno_final.csv')
green_cleaned = pd.read_csv('green cleaned new.csv')

In [20]:
# 1. Standardize the identifier column so both match exactly
green_cleaned = green_cleaned.rename(columns={'PERMNO': 'permno'})

# 2. Convert both date columns to proper datetime objects
final_sp500_df['date'] = pd.to_datetime(final_sp500_df['date'])
green_cleaned['datadate'] = pd.to_datetime(green_cleaned['datadate'])

# 3. Create a temporary 'year_month' column in both dataframes to act as a bridge
# This groups every daily date into its corresponding month (e.g., '2023-10')
final_sp500_df['year_month'] = final_sp500_df['date'].dt.to_period('M')
green_cleaned['year_month'] = green_cleaned['datadate'].dt.to_period('M')

# 4. Isolate just the columns you want to bring over to prevent clutter
returns_subset = green_cleaned[['permno', 'year_month', 'ret_fwd_1']]

# 5. Perform a Left Merge
# This keeps all your daily news rows and attaches the return for that specific month
merged_df = final_sp500_df.merge(
    returns_subset, 
    on=['permno', 'year_month'], 
    how='left'
)

# 6. Clean up by dropping the temporary bridging column
merged_df = merged_df.drop(columns=['year_month'])

# merged_df now contains your daily news with the corresponding monthly 'ret_fwd_1' attached

In [21]:
merged_df

,rp_entity_id,rpa_date_utc,timestamp_utc,rp_story_id,entity_name,event_sentiment_score,event_relevance,event_similarity_days,category,news_type,headline,cusip,clean_headline,et_time,date,permno,ticker,ret_fwd_1
0,D6489C,2023-04-13,2023-04-13 00:46:54.198000+00:00,B707EE9390075FF374726D28E0BF47BA,Oracle Corp.,-0.49,100,202.63656,insider-sell,FULL-ARTICLE,"CEO Catz Sells 3,075,942 Of Oracle Corp >ORCL",68389X105,"CEO Catz Sells 3,075,942 Of Oracle Corp >ORCL",2023-04-12 20:46:54.198000-04:00,2023-04-13,10104,ORCL,0.118454
1,D6489C,2023-04-18,2023-04-18 12:35:02.092000+00:00,7D31EA0ADA010514B28D60ED7A754F6E,Oracle Corp.,0.49,100,365.00000,partnership,PRESS-RELEASE,Deloitte and Oracle Collaborate to Deliver Bet...,68389X105,Deloitte and Oracle Collaborate to Deliver Bet...,2023-04-18 08:35:02.092000-04:00,2023-04-18,10104,ORCL,0.118454
2,D6489C,2023-04-18,2023-04-18 13:00:04.524000+00:00,BEB473094521697890D7F140799238F0,Oracle Corp.,0.49,100,365.00000,partnership,PRESS-RELEASE,GitLab and Oracle Expand Strategic Collaborati...,68389X105,GitLab and Oracle Expand Strategic Collaborati...,2023-04-18 09:00:04.524000-04:00,2023-04-18,10104,ORCL,0.118454
3,D6489C,2023-04-18,2023-04-18 13:00:24.949000+00:00,7B291A9DE39667074406261454903187,Oracle Corp.,0.49,100,365.00000,partnership,PRESS-RELEASE,Oracle and Zoom Join Forces to Expand Access t...,68389X105,Oracle and Zoom Join Forces to Expand Access t...,2023-04-18 09:00:24.949000-04:00,2023-04-18,10104,ORCL,0.118454
4,D6489C,2023-05-02,2023-05-02 12:30:00.934000+00:00,AF6E9BC4B54104E8CB685EF192111D48,Oracle Corp.,0.49,100,365.00000,partnership,PRESS-RELEASE,Paymentus and Oracle Partner for Integrated Bi...,68389X105,Paymentus and Oracle Partner for Integrated Bi...,2023-05-02 08:30:00.934000-04:00,2023-05-02,10104,ORCL,0.124127
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44836,DD3BB1,2025-11-04,2025-11-04 09:15:04.442000+00:00,8ACD7AFFD6521DA1DFC0F18FFBECE95A,Tesla Inc.,0.02,100,365.00000,stock-price,FULL-ARTICLE,Tesla Shares Slide Premarket After Major Inves...,88160R101,Tesla Shares Slide Premarket After Major Inves...,2025-11-04 04:15:04.442000-05:00,2025-11-04,93436,TSLA,0.045447
44837,DD3BB1,2025-11-10,2025-11-10 12:24:27.419000+00:00,184A66C7B549724D08C2E298DEB7DD79,Tesla Inc.,0.00,100,365.00000,stock-price,FULL-ARTICLE,Tesla Stock Rises as Cybertruck Leader Exits. ...,88160R101,Tesla Stock Rises as Cybertruck Leader Exits. ...,2025-11-10 07:24:27.419000-05:00,2025-11-10,93436,TSLA,0.045447
44838,DD3BB1,2025-12-13,2025-12-13 02:31:14.731000+00:00,7E890DAE42E32C83531FCDCD487A15F3,Tesla Inc.,0.62,100,365.00000,acquisition-acquirer,FULL-ARTICLE,"Cathie Wood's ARK Sold Tesla, Bought Chinese T...",88160R101,"Cathie Wood's ARK Sold Tesla, Bought Chinese T...",2025-12-12 21:31:14.731000-05:00,2025-12-15,93436,TSLA,NaN
44839,DD3BB1,2025-12-15,2025-12-15 13:36:34.462000+00:00,260AAD4040FDD255F72C01EF152E8E20,Tesla Inc.,0.39,100,365.00000,business-contract,PRESS-RELEASE,Press Release: SPIE signs a European framework...,88160R101,SPIE signs a European framework agreement with...,2025-12-15 08:36:34.462000-05:00,2025-12-15,93436,TSLA,NaN


In [26]:
merged_df.to_csv('ravenpack_sp500_permno_final.csv', index=False)
